# Sums of Squares in Real Quadratic Rings — Paper Illustrations

This notebook walks through every algorithm and example from the paper.

In [1]:
import os
import sys

repo_root = os.path.abspath(os.path.join(os.getcwd(), '..')) if os.path.basename(os.getcwd()) == 'notebooks' else os.getcwd()
src_path = os.path.join(repo_root, 'src')
if src_path not in sys.path:
    sys.path.insert(0, src_path)


In [2]:
from quadratic_sos.ring import RealQuadraticOrder, QuadraticInteger
from quadratic_sos.enumeration import (
    is_representable, enumerate_search_set, relevant_squares, relevant_squares_with_roots
)
from quadratic_sos.length import exact_length, decomposition
from quadratic_sos.pythagoras import pythagoras_number, pythagoras_element, verify_pythagoras_element
from quadratic_sos.tables import enumerate_totally_positive, length_distribution

## 1. Ring arithmetic in O_D

In [3]:
O10 = RealQuadraticOrder(10)
print(f"D = {O10.D}, D mod 4 = {O10.D % 4}")

alpha = (18, 2)
print(f"alpha = {O10.elem(*alpha)}")
print(f"Tr = {O10.trace(alpha)}, N = {O10.norm(alpha)}")
print(f"Totally positive: {O10.is_totally_positive(alpha)}")
print(f"(1+sqrt10)^2 = {O10.elem(*O10.sqr((1,1)))}")

O5 = RealQuadraticOrder(5)
print(f"\nD=5: omega^2 = {O5.elem(*O5.sqr((0,1)))}")
print(f"N(3+omega_5) = {O5.norm((3,1))}, Tr = {O5.trace((3,1))}")

D = 10, D mod 4 = 2
alpha = 18 + 2·√10
Tr = 36, N = 284
Totally positive: True
(1+sqrt10)^2 = 11 + 2·√10

D=5: omega^2 = 1 + ω_5
N(3+omega_5) = 11, Tr = 7


## 2. Peters' representability criterion (Algorithm 1)

In [4]:
print("3+sqrt10 repr?", is_representable(O10, (3, 1)))
print("18+2sqrt10 repr?", is_representable(O10, (18, 2)))
print("-1 repr?", is_representable(O10, (-1, 0)))
print()
for n in range(1, 11):
    print(f"  {n}: {is_representable(O10, (n, 0))}")

3+sqrt10 repr? False
18+2sqrt10 repr? True
-1 repr? False

  1: True
  2: True
  3: True
  4: True
  5: True
  6: True
  7: True
  8: True
  9: True
  10: True


## 3. Square enumeration — the search set (Algorithm 2)

In [5]:
alpha = (18, 2)
roots = enumerate_search_set(O10, alpha)
print(f"B(alpha) has {len(roots)} elements:")
for r in sorted(roots):
    print(f"  x = {O10.elem(*r)},  x^2 = {O10.elem(*O10.sqr(r))}")

S = relevant_squares(O10, alpha)
print(f"\nS(alpha) has {len(S)} distinct squares:")
for sq in S:
    print(f"  {O10.elem(*sq)}")

B(alpha) has 11 elements:
  x = -3,  x^2 = 9
  x = -2,  x^2 = 4
  x = -1 - √10,  x^2 = 11 + 2·√10
  x = -1,  x^2 = 1
  x = -√10,  x^2 = 10
  x = 0,  x^2 = 0
  x = √10,  x^2 = 10
  x = 1,  x^2 = 1
  x = 1 + √10,  x^2 = 11 + 2·√10
  x = 2,  x^2 = 4
  x = 3,  x^2 = 9

S(alpha) has 6 distinct squares:
  0
  1
  4
  9
  10
  11 + 2·√10


## 4. Exact integral length (Algorithm 3)

In [6]:
cases = [((1,0),"1"), ((2,0),"2"), ((3,0),"3"), ((7,0),"7"),
         ((18,2),"18+2sqrt10"), ((3,1),"3+sqrt10")]
for pair, name in cases:
    ell = exact_length(O10, pair)
    print(f"  ell({name}) = {ell if ell is not None else 'inf'}")

  ell(1) = 1
  ell(2) = 2
  ell(3) = 3
  ell(7) = 4
  ell(18+2sqrt10) = 5
  ell(3+sqrt10) = inf


## 5. Constructive decompositions

In [7]:
def show_dec(order, alpha, name):
    dec = decomposition(order, alpha)
    if dec is None:
        print(f"{name}: not representable"); return
    terms = " + ".join(f"({order.elem(*r)})^2" for r in dec)
    print(f"{name} = {terms}")
    total = (0,0)
    for r in dec:
        total = order.add(total, order.sqr(r))
    assert total == alpha
    print(f"  verified, length {len(dec)}")

for a, n in [((1,0),"1"),((2,0),"2"),((3,0),"3"),((7,0),"7"),((10,0),"10"),((11,2),"11+2sqrt10")]:
    show_dec(O10, a, n)

1 = (-1)^2
  verified, length 1
2 = (-1)^2 + (-1)^2
  verified, length 2
3 = (-1)^2 + (-1)^2 + (-1)^2
  verified, length 3
7 = (-1)^2 + (-1)^2 + (-1)^2 + (-2)^2
  verified, length 4
10 = (-√10)^2
  verified, length 1
11+2sqrt10 = (-1 - √10)^2
  verified, length 1


## 6. Pythagoras numbers and witnesses (Algorithm 4)

In [8]:
for D in [2, 3, 5, 6, 7, 10, 11, 13, 17, 21, 29, 37]:
    O = RealQuadraticOrder(D)
    p, w, ell = verify_pythagoras_element(O)
    ok = "ok" if ell == p else "FAIL"
    print(f"D={D:>3}: P={p}, witness={O.elem(*w)}, computed length={ell} [{ok}]")

D=  2: P=3, witness=6 + 2·√2, computed length=3 [ok]
D=  3: P=3, witness=9 + 4·√3, computed length=3 [ok]
D=  5: P=3, witness=3 + ω_5, computed length=3 [ok]
D=  6: P=4, witness=10 + 2·√6, computed length=4 [ok]
D=  7: P=4, witness=11 + 2·√7, computed length=4 [ok]
D= 10: P=5, witness=18 + 2·√10, computed length=5 [ok]
D= 11: P=5, witness=19 + 2·√11, computed length=5 [ok]
D= 13: P=5, witness=10 + 4·ω_13, computed length=5 [ok]
D= 17: P=5, witness=11 + ω_17, computed length=5 [ok]
D= 21: P=5, witness=12 + ω_21, computed length=5 [ok]
D= 29: P=5, witness=14 + ω_29, computed length=5 [ok]
D= 37: P=5, witness=16 + ω_37, computed length=5 [ok]


## 7. Reproducing Table 1: O_10, Tr <= 40

In [9]:
counts, first = length_distribution(O10, 40)
print(f"Total: {sum(counts.values())}")
for key in [1, 2, 3, 4, 5, None]:
    label = "inf" if key is None else str(key)
    ex = str(O10.elem(*first[key])) if key in first else "-"
    print(f"  length {label:>3}: count {counts[key]:>3}, first = {ex}")

Total: 134
  length   1: count  11, first = 1
  length   2: count  22, first = 2
  length   3: count  14, first = 3
  length   4: count   9, first = 7
  length   5: count   2, first = 18 - 2·√10
  length inf: count  76, first = 4 - √10


## 8. Multi-field comparison, Tr <= 30

In [10]:
for D in [2, 3, 5, 6, 7, 10, 11, 13]:
    O = RealQuadraticOrder(D)
    c, _ = length_distribution(O, 30)
    tot = sum(c.values())
    row = [c.get(k,0) for k in [1,2,3,4,5,None]]
    print(f"D={D:>3} P={O.pythagoras_number()} #tp={tot:>4}: " + " ".join(f"l{k if k else 'inf'}={v}" for k,v in zip([1,2,3,4,5,"inf"],row)))

D=  2 P=3 #tp= 169: l1=15 l2=50 l3=20 l4=0 l5=0 linf=84
D=  3 P=3 #tp= 139: l1=13 l2=35 l3=23 l4=0 l5=0 linf=68
D=  5 P=3 #tp= 207: l1=22 l2=92 l3=93 l4=0 l5=0 linf=0
D=  6 P=4 #tp=  99: l1=10 l2=19 l3=14 l4=4 l5=0 linf=52
D=  7 P=4 #tp=  91: l1=8 l2=17 l3=10 l4=6 l5=0 linf=50
D= 10 P=5 #tp=  75: l1=8 l2=12 l3=6 l4=3 l5=0 linf=46
D= 11 P=5 #tp=  73: l1=8 l2=9 l3=5 l4=3 l5=0 linf=48
D= 13 P=5 #tp= 129: l1=12 l2=41 l3=48 l4=14 l5=2 linf=12


## 9. Parallelogram vs bounding-box efficiency

In [11]:
from decimal import Decimal, getcontext, ROUND_FLOOR
getcontext().prec = 80

def box_vs_para(order, alpha):
    D = order.D; sqrtD = Decimal(D).sqrt()
    w1 = (Decimal(1)+sqrtD)/2 if order.is_mod1 else sqrtD
    w2 = (Decimal(1)-sqrtD)/2 if order.is_mod1 else -sqrtD
    delta = abs(w1-w2); a,b = alpha
    s1 = Decimal(a)+Decimal(b)*w1; s2 = Decimal(a)+Decimal(b)*w2
    A1,A2 = s1.sqrt(),s2.sqrt()
    U = int(((abs(w1)*A2+abs(w2)*A1)/delta).to_integral_value(rounding=ROUND_FLOOR))
    V = int(((A1+A2)/delta).to_integral_value(rounding=ROUND_FLOOR))
    return (2*U+1)*(2*V+1), len(enumerate_search_set(order, alpha))

for D, alpha in [(10,(18,2)),(10,(100,6)),(13,(50,4)),(7,(30,2)),(2,(20,4)),(3,(30,6))]:
    O = RealQuadraticOrder(D)
    if not O.is_totally_positive(alpha): continue
    box, act = box_vs_para(O, alpha)
    print(f"D={D:>3} {str(O.elem(*alpha)):>16}: box={box:>5} |B|={act:>4} ratio={act/box:.2f}")

D= 10       18 + 2·√10: box=   27 |B|=  11 ratio=0.41
D= 10      100 + 6·√10: box=  133 |B|=  61 ratio=0.46
D= 13      50 + 4·ω_13: box=  105 |B|=  57 ratio=0.54
D=  7        30 + 2·√7: box=   55 |B|=  21 ratio=0.38
D=  2        20 + 4·√2: box=   63 |B|=  25 ratio=0.40
D=  3        30 + 6·√3: box=   77 |B|=  31 ratio=0.40
